In [26]:
import os
import pandas as pd # type: ignore
import numpy as np # type: ignore
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pickle

from gglasso.problem import glasso_problem
from gglasso.helper.utils import sparsity
from gglasso.helper.model_selection import ebic
from tqdm.notebook import tqdm

try:
    os.chdir('/container/mount/point')
except FileNotFoundError:
    print("Warning: Directory '/container/mount/point' does not exist.")

from utils.helper import perform_bh_correction_and_filter
from utils.helper import calculate_correlation
from utils.helper import transform_features, scale_array_by_diagonal
from utils.solver import ADMM_single

In [27]:
def create_and_select_model(model, corr, sample_ids, lambda1, mu1_range=None):
    """
    Create and select a graphical lasso model based on the specified parameters.

    Parameters:
    - model (str): The type of model to create and select ("sparse" or "low_rank").
    - corr (float): The correlation matrix.
    - sample_ids (list): List of sample IDs.
    - lambda1 (float): The lambda1 parameter for model selection.
    - mu1_range (float or None): The mu1_range parameter for model selection (only for "low_rank" model).

    Returns:
    - P: The graphical lasso problem object after model creation and selection.
    """
    if "sparse" in model:
        P = glasso_problem(corr, N=len(sample_ids), latent=False)
        modelselect_params = {'lambda1_range': lambda1}
    elif "low_rank" in model:
        P = glasso_problem(corr, N=len(sample_ids), latent=True)
        modelselect_params = {'lambda1_range': lambda1, 'mu1_range': mu1_range}
    else:
        raise ValueError("Invalid model type")

    P.model_selection(modelselect_params=modelselect_params, method='eBIC')
    
    return P

In [ ]:
def calculate_sparsity_levels(result):
    """
    Calculate sparsity levels based on the given result dictionary.

    Parameters:
    - result (dict): Dictionary containing experiment results.

    Returns:
    - list: List of sparsity levels calculated from the result dictionary.
    """
    sp_levels = []

    for key, item in result.items():
        l1_list = result[key]['stats']['BEST']['lambda1']
        l1_best = result[key]['stats']['LAMBDA']
        
        l1_ix = np.where(l1_list == l1_best)[0]
        
        sp_best = result[key]['stats']['SP'][l1_ix].item()
        
        sp_levels.append(sp_best)

    return sp_levels

### KORA Dataset

In [28]:
# Load your  matched sample dataframe and ASV table
kora_matched_df = pd.read_csv("data/smoking_KORA_experiment.csv", index_col=0)
asv = pd.read_csv("data/filtered_count_table.csv", index_col=0)
simulated_outcomes = pd.read_csv("data/simulated_KORA_outcomes.csv", index_col=0)
taxa = pd.read_csv('data/taxonomy_clean.csv', index_col=0)
print(f"ASV table shape (features, samples): {asv.shape}")


with open("data/taxa_dict.pkl", "rb") as f:
    taxa_dict = pickle.load(f)

ASV table shape (features, samples): (1469, 436)


In [ ]:
# Ensure sample IDs match and order
sample_ids = kora_matched_df.index.astype(str)
asv = asv.loc[:, sample_ids]

lambda1 = np.logspace(-3, 0, 5)
result_dict = {}

output_dir = "data/network_results"
os.makedirs(output_dir, exist_ok=True)

for level, data in taxa_dict.items():
    print(f"Calculating correlations and solving sparse model for: {level} with data shape: {data.shape}")

    # Get smoker and non-smoker sample IDs
    smoker_ids = kora_matched_df[kora_matched_df['smoking_bin'] == 1].index.astype(str)
    nonsmoker_ids = kora_matched_df[kora_matched_df['smoking_bin'] == 0].index.astype(str)

    # Select ASV data for each group at the current taxonomic level
    data_smoker = data.loc[:, smoker_ids]
    data_nonsmoker = data.loc[:, nonsmoker_ids]

    # Compute correlation matrices for each group
    corr_smoker = calculate_correlation(data_smoker, smoker_ids)
    corr_nonsmoker = calculate_correlation(data_nonsmoker, nonsmoker_ids)

    # Solve sparse model for true network
    P_smoker = create_and_select_model("sparse", corr_smoker, smoker_ids, lambda1)
    P_nonsmoker = create_and_select_model("sparse", corr_nonsmoker, nonsmoker_ids, lambda1)

    result_dict[level] = {
        "P_smoker_true": P_smoker,
        "P_nonsmoker_true": P_nonsmoker
    }

    # --- Simulated outcomes ---
    n_iter = 1000
    simulated_result = {"smoker": [], "nonsmoker": []}
    W_frame = simulated_outcomes.iloc[:, -n_iter:]

    for i in range(n_iter):
        w_tmp = W_frame.iloc[:, i].to_frame(name="w")
        w_tmp.index = w_tmp.index.astype(str)

        # For smokers
        sim_smoker_ids = w_tmp[w_tmp["w"] == 1].index.intersection(smoker_ids)
        sim_data_smoker = data.loc[:, sim_smoker_ids]
        if sim_data_smoker.shape[1] > 1:
            corr_sim_smoker = calculate_correlation(sim_data_smoker, sim_smoker_ids)
            P_sim_smoker = create_and_select_model("sparse", corr_sim_smoker, sim_smoker_ids, lambda1)
            simulated_result["smoker"].append(P_sim_smoker)

        # For non-smokers
        sim_nonsmoker_ids = w_tmp[w_tmp["w"] == 0].index.intersection(nonsmoker_ids)
        sim_data_nonsmoker = data.loc[:, sim_nonsmoker_ids]
        if sim_data_nonsmoker.shape[1] > 1:
            corr_sim_nonsmoker = calculate_correlation(sim_data_nonsmoker, sim_nonsmoker_ids)
            P_sim_nonsmoker = create_and_select_model("sparse", corr_sim_nonsmoker, sim_nonsmoker_ids, lambda1)
            simulated_result["nonsmoker"].append(P_sim_nonsmoker)

    result_dict[level]["simulated"] = simulated_result

    # Save result_dict for this level
    with open(f"{output_dir}/result_dict_{level}.pkl", "wb") as f:
        pickle.dump(result_dict[level], f)

Calculating correlations and solving sparse model for: domain with data shape: (2, 436)
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
Calculating correlations and solving sparse model for: phylum with data shape: (9, 436)
Calculating correlations and solving sparse model for: class with data shape: (15, 436)
Calculating correlations and solving sparse model for: order with data shape: (41, 436)
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
ADMM terminated after 23 iterations with status: optimal.
Calculating correlations and solving sparse model for: family with d

### Differential network analysis

In [ ]:
with open('data/P_{0}_{1}_{2}_{3}.pickle'.format(model, "allergy", dataset, level), 'rb') as file:
    alg_sol = pickle.load(file)

with open('data/P_{0}_{1}_{2}_{3}.pickle'.format(model, "control", dataset, level), 'rb') as file:
    ctrl_sol = pickle.load(file)

#### True network

In [ ]:
theta_alg = alg_sol['oracle']['precision']
theta_ctrl = ctrl_sol['oracle']['precision']


names = counts.index.values
mask = np.tril(np.ones(theta_ctrl.shape), k=-1)


alg_oracle = pd.DataFrame(theta_alg, index=names, columns=names)
ctrl_oracle = pd.DataFrame(theta_ctrl, index=names, columns=names)

# Calculate the difference
obs_stat = abs(alg_oracle - ctrl_oracle)

# Take only the lower triangle part since the matrix is symmetric
obs_stat = (obs_stat * mask).stack()

# Drop edges where there is no difference
obs_stat = obs_stat[obs_stat != 0]

### TO DO: change to a 10e-4
# Drop if the diff is less 0.05 
obs_stat = obs_stat[obs_stat >= 10e-4]

#### Permuted networks

In [ ]:
#change that identical values do not cancel out
test_stats = list()

for i, key in enumerate(alg_sol.keys(), 0):
    if 'fake' in key:
        key_alg = f"{dataset}_fake_allergy_{i}"
        key_ctrl = f"{dataset}_fake_control_{i}"
        
        alg_rand = alg_sol.get(key_alg, {}).get('precision')
        ctrl_rand = ctrl_sol.get(key_ctrl, {}).get('precision')
        
        if alg_rand is not None and ctrl_rand is not None:
            
            # To check the case when matrices has exactly the same entries
            if np.allclose(alg_rand, ctrl_rand, atol=1e-8):
                print(f"For key '{key}', alg_rand and ctrl_rand are close.")
            else:
                test_stat = np.abs(alg_rand - ctrl_rand)
                test_stats.append(test_stat)

In [ ]:
result_list = [pd.DataFrame(t, index=names, columns=names).stack() for t in test_stats]
test_stats = pd.concat(result_list, axis=1)

# # Select only edges from oracle solution
test_stats = test_stats.loc[obs_stat.index]

#### p-value adjustment

In [ ]:
p_value = calculate_unadjusted_p_values(test_stats, obs_stat, test_type="two-sided")

In [ ]:
fig_pvalue = px.histogram(p_value['p-value'], x="p-value", nbins = 10,  color_discrete_sequence=['#641E62'])
# fig_pvalue.show()
fig_pvalue.write_image("plots/png/pvalues.png")

In [ ]:
min_p_values = min_unadjusted_p_values(test_stats)

In [ ]:
adj_p_values = adjusted_p_values(min_p_values, p_value)

In [ ]:
p_value["Lee et al."] = np.array(adj_p_values)

In [ ]:
# p_value.to_csv('data/adj_diffnet_{0}_{1}_ige.csv'.format("species", test_stats.shape[1]))

check p-values with actual difference

#### Visualization

In [ ]:
n_iter = 2000
level = 'species'

p_value = pd.read_csv('data/adj_diffnet_{0}_{1}_ige.csv'.format(level, n_iter), index_col=0)
df_alg, df_ctrl = pd.read_csv('data/df_alg.csv', index_col=0), pd.read_csv('data/df_ctrl.csv', index_col=0)

In [ ]:
test = p_value.head(11)

In [ ]:
names = np.array(test.reset_index().iloc[:, :2].values).flatten()
names

In [ ]:
# df_alg = alg_oracle.loc[names, names]

# # Remove rows with duplicate labels
# df_alg = df_alg.loc[~df_alg.index.duplicated(keep='first')]

# # Remove columns with duplicate labels
# df_alg = df_alg.loc[:, ~df_alg.columns.duplicated(keep='first')]

In [ ]:
# df_ctrl = ctrl_oracle.loc[names, names]

# # Remove rows with duplicate labels
# df_ctrl = df_ctrl.loc[~df_ctrl.index.duplicated(keep='first')]

# # Remove columns with duplicate labels
# df_ctrl = df_ctrl.loc[:, ~df_ctrl.columns.duplicated(keep='first')]

In [ ]:
from bokeh.plotting import figure, output_notebook, show, output_file, save
from bokeh.io.export import export_svgs
# Enable notebook mode
output_notebook()

width = 3000
height = 3000
label_size = "16pt"

clust_order = _get_order(df_alg)

In [ ]:
test_heatmap = plot_ordered_heatmap(df_alg, order=clust_order, title="species heatmap")

# # Show the plot (this will also save it as an HTML file)
show(test_heatmap)

save(test_heatmap, filename="plots/html/bokeh_plot.html")

# # Export the plot as an SVG image
# export_svgs(test_heatmap, filename="plots/svg/test_heatmap.svg")

In [ ]:
test_heatmap = plot_ordered_heatmap(5*df_ctrl, order=clust_order, title="species heatmap")

# # Show the plot (this will also save it as an HTML file)
show(test_heatmap)

In [ ]:
# test = [ "g__Christensenellaceae_R-7_group__bacterium_YE57", "g__Acidaminococcus__Acidaminococcus_fermentans", "g__CHKCI002__uncultured_bacterium"]
# edges = df_ctrl.loc[test]

G_ctrl = create_graph(df_ctrl, threshold=0.0)

ctrl_net = create_network_visualization(G_ctrl, height=1500, width=1800, show_labels=False, size_degree=True, scale_edge=20, scale_node=17)

ctrl_net.show('plots/ctrl_network.html')

In [ ]:
G_alg = create_graph(df_alg, threshold=0.0)

alg_net = create_network_visualization(G_alg, height=1500, width=1800, show_labels=False, size_degree=True, scale_edge=20, scale_node=17)

alg_net.show('plots/alg_network.html')

HTML and the notebook should be in the same directory!